# Advisor results walkthrough
INTERNAL ONLY: preserved pre-essentials notebook and selected-reference analyses.
One selected matrix labeled Learned, with OTFS and AFDM. Same Learned for every SNR, modulation, receiver and structural view. See docs/SELECTION.md: retained after a post-hoc tradeoff review, not a universal winner or independent selection test.
Provision local artifacts first; no training or BER evaluation in this notebook.


In [ ]:
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[key] = '1'
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from waveforms.core import configure
from waveforms.artifacts import load_reference
from waveforms.delay_doppler import bridge
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'support/configs/model.json').is_file())
artifacts = Path(os.environ.get('ADVISOR_ARTIFACTS', str(root/'local/artifacts')))
if not artifacts.exists():
    raise FileNotFoundError('Provision local bundle first; see README. No old source-project paths are used.')
configure()


## H → W → A
Columns of W synthesize symbols. Original ordering is authoritative; saved group permutation is an additional fixed view.


In [ ]:
data = np.load(artifacts/'focused/data/channels.npz', allow_pickle=False)
w = load_reference(artifacts)
h = torch.from_numpy(data['H'])
a = w.mH @ h @ w
b = bridge(w,h)
print('H,W,A:', tuple(h.shape),tuple(w.shape),tuple(a.shape),a.dtype,a.device)
print('unitarity:',float(torch.linalg.matrix_norm(w.mH@w-torch.eye(64))))
print('complex spectral reconstruction:',float(torch.linalg.matrix_norm(a-b['A_spectral'])/torch.linalg.matrix_norm(a)))
print('paths:',data['tau'],data['nu'], 'normalization:',data['normalization'])


## Core figures
Default25dB view only; full numerical grids retained through30/40dB. Main raw slicer; fixed-Q and full receivers separated. Generate figures with the README command first.


In [ ]:
from IPython.display import display, Image
for name in ['channel_matrices_native','basis_time','basis_frequency','ber_QPSK_raw','ber_16QAM_raw']:
    path = Path(os.environ.get('ADVISOR_FIGURES', str(root/'local/figures')))/f'{name}.png'
    if path.exists(): display(Image(filename=str(path), width=1000))
    else: print('Generate first:',path.name)
summary = pd.read_csv(artifacts/'focused/ber_summary.csv')
display(summary[(summary.snr_db==25)&(summary.slicer=='raw')][['modulation','waveform','receiver','errors','bits','ber','ci_low','ci_high']])


## Selected Learned supplementary evidence
Same verified matrix and baselines under elementary impairments. Structural transfer, not BER or the separate CFO W/R model. Frequency groups are approximate/leaky. All alternative candidates and internal comparisons remain in internal/notebooks/variants.ipynb.


In [ ]:
from waveforms.selection import identity
from waveforms.supplementary import selected_transfer
print(identity())
display(selected_transfer(artifacts).query('strength == 1'))
